# Figures 2 and 3: distributed GP experiments

Run cells in order from a fresh kernel with this project as the working directory. The notebook trains local GP models, plots hyperparameter convergence, and measures consensus error against plaintext PoE. Figures are saved to `img/hyp.pdf` and `img/accuracy.pdf`. See `README.md` for installation and experiment settings.

In [ ]:
import torch
import gpytorch
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

try:
    from tqdm.auto import tqdm
except ImportError:

    def tqdm(iterable=None, **kwargs):
        return iterable


from share import privacy_preserving_avg_consensus
from hypOpt import hyperparameter_optimization, hyperparameter_optimization_1D
from plot import plot_hyperparameter, plot_accuracy
import os
import psutil
import platform
import time

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

%load_ext autoreload
%autoreload 2

In [ ]:
seed = 1

np.random.seed(seed)
torch.manual_seed(seed)


print("📦 Versions:")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"GPyTorch: {gpytorch.__version__}")


print("\n🚀 GPU Info:")
if torch.cuda.is_available():
    print("CUDA available: True")
    for i in range(torch.cuda.device_count()):
        print(f"  [{i}] {torch.cuda.get_device_name(i)}")
else:
    print("CUDA available: False")


print("\n🧮 CPU Info:")
print(f"Processor: {platform.processor()}")
print(f"Cores (physical): {psutil.cpu_count(logical=False)}")
print(f"Cores (logical): {psutil.cpu_count(logical=True)}")


print("\n🧠 RAM Info:")
vmem = psutil.virtual_memory()
print(f"Total: {vmem.total / (1024**3):.2f} GB")
print(f"Available: {vmem.available / (1024**3):.2f} GB")


print("\n⚙️ CUDA Toolkit Version:")
try:
    output = os.popen("nvcc --version").read()
    print(output if output else "nvcc not found")
except:
    print("Could not retrieve CUDA version")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Figure 2: hyperparameter convergence

Each agent alternates one secure consensus step with local gradient ascent. Separate GP models are trained for the seven SARCOS outputs; the figure shows the first output and Diabetes.

### Configuration

Use 20 agents with four neighbors each, 30 training iterations, and 20 prediction-consensus iterations. The training scale is $L_z=2^{-20}$; the Fig. 3 sweep sets its own prediction scales.

In [ ]:
config = {
    "M": 20,  # Number of agents
    "Ni": 4,  # Number of neighbors
    "q": 2**40,  # Modulus
    "Lw": 1 / 40,  # Scale factor
    "Lz": 1 / 2**20,  # Hyperparameter-optimization scale factor
    "Thyp": 30,  # Optimization step for hyperparameter optimization
    "eta": 0.1,  # Learning rate
    "decay": 0.99,  # Decay rate
    "T": 20,  # Optimization step for main Protocol
    "communication_latency": 0.0002,  # 0.2 ms per communication
}

### Network topology

#### Graph

Construct a connected regular graph in which every edge has a common neighbor, as required by Assumption 2.

In [ ]:
import networkx as nx

# Generate an undirected, connected Ni-regular graph satisfying
# Assumption 2. Every agent has exactly Ni neighbors.
assert 1 <= config["Ni"] < config["M"], "Ni must satisfy 1 <= Ni < M"
assert (
    config["M"] * config["Ni"]
) % 2 == 0, "An M-node Ni-regular undirected graph requires M*Ni to be even"

if config["Ni"] == config["M"] - 1:
    # The complete graph is Ni-regular and every edge lies in a triangle.
    G = nx.complete_graph(config["M"])
elif config["Ni"] >= 4 and config["Ni"] % 2 == 0:
    # A cycle power has degree 2*radius. Radius >= 2 ensures that every
    # edge lies in a triangle, so Assumption 2 holds by construction.
    ring_radius = config["Ni"] // 2
    G = nx.power(nx.cycle_graph(config["M"]), ring_radius)
else:
    # For other feasible degrees, search reproducibly among regular graphs.
    graph_rng = np.random.default_rng(seed)
    max_graph_attempts = 10_000
    for graph_attempt in range(1, max_graph_attempts + 1):
        graph_seed = int(graph_rng.integers(0, 2**32 - 1))
        candidate = nx.random_regular_graph(
            d=config["Ni"], n=config["M"], seed=graph_seed
        )
        candidate_violations = [
            (i, j)
            for i, j in candidate.edges()
            if not (set(candidate.neighbors(i)) & set(candidate.neighbors(j)))
        ]
        if nx.is_connected(candidate) and not candidate_violations:
            G = candidate
            break
    else:
        raise RuntimeError(
            "Could not find a connected Ni-regular graph satisfying "
            f"Assumption 2 after {max_graph_attempts} attempts."
        )

degrees = dict(G.degree())
assumption_2_violations = [
    (i, j) for i, j in G.edges() if not (set(G.neighbors(i)) & set(G.neighbors(j)))
]
assert nx.is_connected(G)
assert all(degree == config["Ni"] for degree in degrees.values())
assert not assumption_2_violations

print(f"Connected: {nx.is_connected(G)}")
print(f"All agent degrees equal Ni={config['Ni']}: True")
print("Degree sequence:", sorted(degrees.values()))
print("Assumption 2 satisfied: True")

#### Metropolis weights

Compute $W$ and verify that $\bar W=W/L_w$ has integer entries.

In [ ]:
# Define Metropolis weights
C = 2
W = np.zeros((config["M"], config["M"]))
for i in range(config["M"]):
    d_i = G.degree[i]
    for j in G.neighbors(i):
        d_j = G.degree[j]
        W[i, j] = 1 / (C * (1 + max(d_i, d_j)))
    W[i, i] = 1 - np.sum(W[i])  # self-weight

config["Wbar"] = W / config["Lw"]

# Equation (9) requires the scaled Metropolis weights to be integers.
Wbar_rounded = np.rint(config["Wbar"])
Wbar_is_integer = np.allclose(config["Wbar"], Wbar_rounded, atol=1e-12)

print("Wbar =")
print(config["Wbar"])
print(f"Wbar contains only integers: {Wbar_is_integer}")

if not Wbar_is_integer:
    invalid_positions = np.argwhere(
        ~np.isclose(config["Wbar"], Wbar_rounded, atol=1e-12)
    )
    invalid_values = [
        (int(i), int(j), float(config["Wbar"][i, j])) for i, j in invalid_positions
    ]
    raise ValueError(
        "Equation (9) is violated: Wbar contains non-integer entries. "
        f"Invalid (row, column, value) entries: {invalid_values}. "
        "Choose a graph degree pattern compatible with Lw or change Lw."
    )

# Store exact integer weights after validation.
config["Wbar"] = Wbar_rounded.astype(np.int64)

### Training and test data

#### SARCOS

Select 4,000 observations from a random training permutation and partition them among the agents. Each observation has 21 inputs and seven outputs. The prediction cells use the first 200 observations of the supplied test file.

In [ ]:
from scipy.io import loadmat


train_data = loadmat("data/sarcos_inv.mat")
full_train = train_data["sarcos_inv"]  # shape: (44484, 28)
test_data = loadmat("data/sarcos_inv_test.mat")
full_test = test_data["sarcos_inv_test"]  # shape: (4449, 28)


sarcos_X_train = full_train[:, :21]
sarcos_Y_train = full_train[:, 21:]

sarcos_X_test = full_test[:, :21]
sarcos_Y_test = full_test[:, 21:]

print("Train x size:", sarcos_X_train.shape)
print("Test x size:", sarcos_X_test.shape)
print("Train y size:", sarcos_Y_train.shape)
print("Test y size:", sarcos_Y_test.shape)


config["k"] = sarcos_Y_train.shape[1]


sarcos_X_train_t = torch.tensor(sarcos_X_train, dtype=torch.float32)
sarcos_Y_train_t = torch.tensor(sarcos_Y_train, dtype=torch.float32)
sarcos_X_test_t = torch.tensor(sarcos_X_test, dtype=torch.float32)
sarcos_Y_test_t = torch.tensor(sarcos_Y_test, dtype=torch.float32)

# Shuffle training observations before partitioning.
sarcos_perm = torch.randperm(sarcos_X_train_t.size(0))[:4000]
sarcos_X_train_shuffled = sarcos_X_train_t[sarcos_perm]
sarcos_Y_train_shuffled = sarcos_Y_train_t[sarcos_perm]

# Partition the training observations among M agents.
sarcos_X_parts = [
    part.to(device) for part in torch.tensor_split(sarcos_X_train_shuffled, config["M"])
]
sarcos_Y_parts = [
    part.to(device) for part in torch.tensor_split(sarcos_Y_train_shuffled, config["M"])
]

#### Diabetes

Use an 80/20 split with `random_state=42`: 353 training observations and 89 test observations. Shuffle the training observations before partitioning.

In [ ]:
diabetes = load_diabetes()
diab_X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
diab_Y = pd.Series(diabetes.target, name="target")

# Split into train/test points (80/20)
diab_X_train, diab_X_test, diab_Y_train, diab_Y_test = train_test_split(
    diab_X, diab_Y, test_size=0.2, random_state=42
)


diab_ntest = diab_X_test.shape[0]

print("Train x size:", diab_X_train.shape)
print("Test x size:", diab_X_test.shape)
print("Train y size:", diab_Y_train.shape)
print("Test y size:", diab_Y_test.shape)


diab_X_train_t = torch.tensor(diab_X_train.values, dtype=torch.float32)
diab_Y_train_t = torch.tensor(diab_Y_train.values, dtype=torch.float32)
diab_X_test_t = torch.tensor(diab_X_test.values, dtype=torch.float32)
diab_Y_test_t = torch.tensor(diab_Y_test.values, dtype=torch.float32)

# Shuffle training observations before partitioning.
diab_perm = torch.randperm(diab_X_train_t.size(0))
diab_X_train_shuffled = diab_X_train_t[diab_perm]
diab_Y_train_shuffled = diab_Y_train_t[diab_perm]

# Partition the training observations among M agents.
diab_X_parts = [
    x.to(device) for x in torch.tensor_split(diab_X_train_shuffled, config["M"])
]
diab_Y_parts = [
    y.to(device) for y in torch.tensor_split(diab_Y_train_shuffled, config["M"])
]

# Keep targets shaped (observations, 1) for the optimizer.
diab_Y_parts = [y.unsqueeze(-1) for y in diab_Y_parts]

### Hyperparameter optimization

Initialize lengthscale, signal variance, and noise variance around the supplied centers, then apply 30 updates with step size $0.1\,0.99^t$. Save the parameter histories for Fig. 2.

#### SARCOS

In [ ]:
sarcos_scales = [10.0, 10.0, 10.0]
sarcos_models_all, sarcos_likelihoods_all, sarcos_history_all, sarcos_loss_sum_all = (
    hyperparameter_optimization(
        sarcos_X_parts,
        sarcos_Y_parts,
        G,
        config,
        sarcos_scales,
        device,
        show_progress=True,
    )
)

for i in range(config["M"]):
    for d in range(config["k"]):
        sarcos_models_all[i][d] = sarcos_models_all[i][d].to(device)
        sarcos_likelihoods_all[i][d] = sarcos_likelihoods_all[i][d].to(device)

#### Diabetes and Figure 2

In [ ]:
diab_scales = [10.0, 10.0, 20.0]
diab_models, diab_likelihoods, diab_history, diab_loss_sum, diab_elapsed = (
    hyperparameter_optimization_1D(
        diab_X_parts,
        diab_Y_parts,
        G,
        config,
        diab_scales,
        device,
        progress_desc="Diabetes hyperparameters",
        show_progress=True,
    )
)
for i in range(config["M"]):
    diab_models[i] = diab_models[i].to(device)
    diab_likelihoods[i] = diab_likelihoods[i].to(device)

In [ ]:
# Show the first SARCOS output and the Diabetes model.

plot_hyperparameter(sarcos_history_all[0], diab_history, config, use_latex=False)

## Figure 3: consensus accuracy

Compare the proposed mean and variance with plaintext product-of-experts (PoE) predictions. For each agent, sum squared errors over output dimensions, average over test points, take the square root, then average over agents.

### Plaintext PoE reference

Time local prediction and PoE aggregation separately over 20 runs.

#### SARCOS

In [ ]:
sarcos_test_samples = 200  # desired number of test points (less than or equal to 4449)

device = next(sarcos_models_all[0][0].parameters()).device

sarcos_X_test_red = sarcos_X_test[:sarcos_test_samples, :]
sarcos_Y_test_red = sarcos_Y_test[:sarcos_test_samples, :]


sarcos_X_test_t_red = torch.tensor(
    sarcos_X_test_red, dtype=torch.float32, device=device
)
sarcos_Y_test_t_red = torch.tensor(
    sarcos_Y_test_red, dtype=torch.float32, device=device
)

# sarcos_means[i][j]: predicted mean from agent i for output dimension j
# sarcos_variances[i][j]: predicted variance from agent i for output dimension j
sarcos_means = [[None for _ in range(config["k"])] for _ in range(config["M"])]
sarcos_variances = [[None for _ in range(config["k"])] for _ in range(config["M"])]

num_repeats = 20
sarcos_local_prediction_times = []

for _ in tqdm(range(num_repeats), desc="SARCOS local prediction", unit="run"):
    start_time = time.time()

    for i in range(config["M"]):
        for j in range(config["k"]):
            model = sarcos_models_all[i][j]
            likelihood = sarcos_likelihoods_all[i][j]
            model.eval()
            likelihood.eval()
            with torch.no_grad(), gpytorch.settings.fast_pred_var():
                pred = likelihood(model(sarcos_X_test_t_red))
                sarcos_means[i][j] = pred.mean  # shape: (N_test,)
                sarcos_variances[i][j] = pred.variance  # shape: (N_test,)

    end_time = time.time()
    sarcos_local_prediction_times.append(end_time - start_time)

mean_time = np.mean(sarcos_local_prediction_times)
std_time = np.std(sarcos_local_prediction_times)

print(
    f"Average time over {num_repeats} runs for local prediction: {mean_time:.6f} ± {std_time:.6f} seconds"
)

In [ ]:
sarcos_poe_means = torch.zeros(sarcos_test_samples, config["k"], device=device)
sarcos_poe_vars = torch.zeros(sarcos_test_samples, config["k"], device=device)

num_repeats = 20
sarcos_poe_aggregation_times = []

# Stack once as (M, N_test, k), then aggregate all outputs together.
sarcos_means_tensor = torch.stack(
    [torch.stack(sarcos_means[m], dim=1) for m in range(config["M"])]
)
sarcos_vars_tensor = torch.stack(
    [torch.stack(sarcos_variances[m], dim=1) for m in range(config["M"])]
)

for _ in tqdm(range(num_repeats), desc="SARCOS PoE aggregation", unit="run"):
    start_time = time.time()

    inv_vars = sarcos_vars_tensor.reciprocal()
    sarcos_poe_vars = inv_vars.sum(dim=0).reciprocal()
    sarcos_poe_means = sarcos_poe_vars * (inv_vars * sarcos_means_tensor).sum(dim=0)

    end_time = time.time()
    sarcos_poe_aggregation_times.append(end_time - start_time)

mean_time = np.mean(sarcos_poe_aggregation_times)
std_time = np.std(sarcos_poe_aggregation_times)

print(
    f"Average time over {num_repeats} runs for PoE aggregation: {mean_time:.6f} ± {std_time:.6f} seconds"
)

#### Diabetes

In [ ]:
diab_means = [None for _ in range(config["M"])]
diab_vars = [None for _ in range(config["M"])]

diab_X_test_t = diab_X_test_t.to(device)

num_repeats = 20
diab_local_prediction_times = []

for _ in tqdm(range(num_repeats), desc="Diabetes local prediction", unit="run"):
    start_time = time.time()

    for i in range(config["M"]):
        model = diab_models[i]
        likelihood = diab_likelihoods[i]
        model.eval()
        likelihood.eval()

        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred = likelihood(model(diab_X_test_t))
            diab_means[i] = pred.mean  # shape: (N_test,)
            diab_vars[i] = pred.variance  # shape: (N_test,)

    end_time = time.time()
    diab_local_prediction_times.append(end_time - start_time)

mean_time = np.mean(diab_local_prediction_times)
std_time = np.std(diab_local_prediction_times)

print(
    f"Average time over {num_repeats} runs for local prediction: {mean_time:.6f} ± {std_time:.6f} seconds"
)

In [ ]:
diab_poe_means = torch.zeros_like(diab_means[0], device=device)  # shape: (N_test,)
diab_poe_vars = torch.zeros_like(diab_vars[0], device=device)  # shape: (N_test,)

diab_means_tensor = torch.stack(diab_means, dim=0).to(device)  # shape: (M, N_test)
diab_vars_tensor = torch.stack(diab_vars, dim=0).to(device)  # shape: (M, N_test)

num_repeats = 20
diab_poe_aggregation_times = []

for _ in tqdm(range(num_repeats), desc="Diabetes PoE aggregation", unit="run"):
    start_time = time.time()

    all_means = diab_means_tensor  # shape: (M, N_test)
    all_vars = diab_vars_tensor  # shape: (M, N_test)

    inv_vars = 1.0 / all_vars
    poe_vars = 1.0 / inv_vars.sum(dim=0)  # shape: (N_test,)
    poe_means = poe_vars * (inv_vars * all_means).sum(dim=0)  # shape: (N_test,)

    diab_poe_means[:] = poe_means
    diab_poe_vars[:] = poe_vars

    end_time = time.time()
    diab_poe_aggregation_times.append(end_time - start_time)

mean_time = np.mean(diab_poe_aggregation_times)
std_time = np.std(diab_poe_aggregation_times)

print(
    f"Average time over {num_repeats} runs for PoE aggregation: {mean_time:.6f} ± {std_time:.6f} seconds"
)

### Scale and iteration sweep

`Lz_values` stores reciprocal scales: `[10, 100, 10000]` means $L_z\in\{10^{-1},10^{-2},10^{-4}\}$. Retain states from iterations 0 through 20. Timing uses 20 repeats; the plotted RMSE uses the last repeat.

#### SARCOS

In [ ]:
Lz_values = [10, 100, 10000]
num_Lz = len(Lz_values)
num_repeats = 20
sarcos_protocol_times_by_lz = np.zeros((num_repeats, num_Lz), dtype=float)

# Precompute static tensors once to avoid rebuilding every repeat
sarcos_means_tensor = sarcos_means_tensor.to(device)
sarcos_vars_tensor = sarcos_vars_tensor.to(device)

weighted_means_base = config["M"] * (sarcos_means_tensor / sarcos_vars_tensor)
precisions_base = config["M"] * (1.0 / sarcos_vars_tensor)
combined_base = torch.stack([weighted_means_base, precisions_base], dim=-1)
z_combined_base = combined_base.reshape(config["M"], -1).cpu().numpy()

target_mean = sarcos_poe_means.to(device).unsqueeze(0).unsqueeze(1).unsqueeze(-1)
target_var = sarcos_poe_vars.to(device).unsqueeze(0).unsqueeze(1).unsqueeze(-1)

# Expect p = sarcos_test_samples * k * 2 (mean and var stacked)
expected_p = sarcos_test_samples * config["k"] * 2
if z_combined_base.shape[1] != expected_p:
    raise ValueError(
        f"z_combined_base has dim {z_combined_base.shape[1]}, expected {expected_p}"
    )

sarcos_rmse_means = torch.zeros((num_Lz, config["T"] + 1), device=device)
sarcos_rmse_vars = torch.zeros((num_Lz, config["T"] + 1), device=device)

for repeat in tqdm(range(num_repeats), desc="SARCOS protocol", unit="run"):

    sarcos_pppoe_means = torch.zeros(
        (num_Lz, config["T"] + 1, sarcos_test_samples, config["k"], config["M"]),
        device=device,
    )
    sarcos_pppoe_vars = torch.zeros(
        (num_Lz, config["T"] + 1, sarcos_test_samples, config["k"], config["M"]),
        device=device,
    )

    for lz_idx, Lz in enumerate(Lz_values):
        lz_start_time = time.time()
        config["Lz"] = 1.0 / Lz

        # privacy_preserving_avg_consensus returns length T+1
        consensus_history = privacy_preserving_avg_consensus(
            z_combined_base,
            G,
            config,
            config["T"],
            device,
            comm_delay=config["communication_latency"],
        )
        steps = len(consensus_history)
        if steps != config["T"] + 1:
            raise ValueError(
                f"Consensus history length {steps} does not match expected {config['T']+1}"
            )

        consensus_tensor = (
            torch.stack(consensus_history, dim=0)
            .to(device)
            .reshape(steps, config["M"], sarcos_test_samples, config["k"], 2)
        )

        denoms = consensus_tensor[..., 1]
        means = consensus_tensor[..., 0] / denoms
        vars_ = 1.0 / denoms

        sarcos_pppoe_means[lz_idx] = means.permute(0, 2, 3, 1)
        sarcos_pppoe_vars[lz_idx] = vars_.permute(0, 2, 3, 1)
        sarcos_protocol_times_by_lz[repeat, lz_idx] = time.time() - lz_start_time


for lz_idx, Lz in enumerate(Lz_values):
    print(
        f"Lz={Lz}: protocol time over {num_repeats} runs = "
        f"{sarcos_protocol_times_by_lz[:, lz_idx].mean():.4f} +/- "
        f"{sarcos_protocol_times_by_lz[:, lz_idx].std():.4f} seconds"
    )

diff_mean = sarcos_pppoe_means - target_mean
diff_var = sarcos_pppoe_vars - target_var

# Paper definition: for each agent, sum squared errors over the seven
# output dimensions, average over test samples, take the square root,
# and finally average the resulting RMSE over all agents.
sarcos_rmse_means = torch.sqrt(
    torch.mean(torch.sum(diff_mean**2, dim=3), dim=2)
).mean(dim=2)
sarcos_rmse_vars = torch.sqrt(torch.mean(torch.sum(diff_var**2, dim=3), dim=2)).mean(
    dim=2
)

for lz_idx in range(num_Lz):
    print(
        f"Lz index {lz_idx}: RMSE Mean at last iteration = {sarcos_rmse_means[lz_idx, config['T']]:.6f}, "
        f"RMSE Variance at last iteration = {sarcos_rmse_vars[lz_idx, config['T']]:.6f}"
    )

#### Diabetes and Figure 3

In [ ]:
Lz_values = [10, 100, 10000]
num_Lz = len(Lz_values)
num_repeats = 20
diab_protocol_times_by_lz = np.zeros((num_repeats, num_Lz), dtype=float)

diab_test_samples = diab_X_test.shape[0]

# Precompute all quantities that do not change between benchmark runs.
diab_means_tensor = torch.stack(diab_means, dim=0).to(device)
diab_vars_tensor = torch.stack(diab_vars, dim=0).to(device)
weighted_means = config["M"] * (diab_means_tensor / diab_vars_tensor)
precisions = config["M"] * diab_vars_tensor.reciprocal()
combined = torch.stack([weighted_means, precisions], dim=-1)
z_combined = combined.reshape(config["M"], -1)

for repeat in tqdm(range(num_repeats), desc="Diabetes protocol", unit="run"):

    diab_pppoe_means = torch.zeros(
        (num_Lz, config["T"] + 1, diab_test_samples, config["M"]), device=device
    )
    diab_pppoe_vars = torch.zeros(
        (num_Lz, config["T"] + 1, diab_test_samples, config["M"]), device=device
    )

    for lz_idx, Lz in enumerate(Lz_values):
        lz_start_time = time.time()
        config["Lz"] = 1.0 / Lz
        consensus_history = privacy_preserving_avg_consensus(
            z_combined,
            G,
            config,
            config["T"],
            device,
            comm_delay=config["communication_latency"],
        )
        consensus_tensor = torch.stack(consensus_history).reshape(
            config["T"] + 1, config["M"], diab_test_samples, 2
        )
        denoms = consensus_tensor[..., 1]
        diab_pppoe_means[lz_idx] = (consensus_tensor[..., 0] / denoms).permute(0, 2, 1)
        diab_pppoe_vars[lz_idx] = denoms.reciprocal().permute(0, 2, 1)
        diab_protocol_times_by_lz[repeat, lz_idx] = time.time() - lz_start_time


for lz_idx, Lz in enumerate(Lz_values):
    print(
        f"Lz={Lz}: protocol time over {num_repeats} runs = "
        f"{diab_protocol_times_by_lz[:, lz_idx].mean():.4f} +/- "
        f"{diab_protocol_times_by_lz[:, lz_idx].std():.4f} seconds"
    )

diab_target_mean = diab_poe_means.to(device)[None, None, :, None]
diab_target_var = diab_poe_vars.to(device)[None, None, :, None]
diab_rmse_means = torch.sqrt(
    torch.mean((diab_pppoe_means - diab_target_mean) ** 2, dim=2)
).mean(dim=2)
diab_rmse_vars = torch.sqrt(
    torch.mean((diab_pppoe_vars - diab_target_var) ** 2, dim=2)
).mean(dim=2)

for lz_idx in range(num_Lz):
    print(
        f"Lz index {lz_idx}: RMSE Mean at last iteration = {diab_rmse_means[lz_idx, config['T']]:.6f}, "
        f"RMSE Variance at last iteration = {diab_rmse_vars[lz_idx, config['T']]:.6f}"
    )

In [ ]:
plot_accuracy(
    sarcos_rmse_means,
    sarcos_rmse_vars,
    diab_rmse_means,
    diab_rmse_vars,
    config,
    Lz_values,
    use_latex=False,
)

## Supplementary timing summary

Report total online time for the configured test batch: local prediction plus PoE aggregation or private consensus. Add the measurements for each repeat before computing the mean and population standard deviation. Training and preprocessing are excluded. The 100-iteration topology comparison is in `table1.ipynb`.

In [ ]:
timing_rows = []


def add_timing_row(dataset, configuration, test_points, samples):
    samples = np.asarray(samples, dtype=float)
    timing_rows.append(
        {
            "dataset": dataset,
            "configuration": configuration,
            "test_points": int(test_points),
            "mean_total_seconds": samples.mean(),
            "std_total_seconds": samples.std(),
            "throughput_points_per_second": test_points / samples.mean(),
        }
    )


sarcos_local = np.asarray(sarcos_local_prediction_times)
diab_local = np.asarray(diab_local_prediction_times)
add_timing_row(
    "SARCOS",
    "PoE baseline",
    sarcos_test_samples,
    sarcos_local + np.asarray(sarcos_poe_aggregation_times),
)
add_timing_row(
    "Diabetes",
    "PoE baseline",
    diab_test_samples,
    diab_local + np.asarray(diab_poe_aggregation_times),
)

for lz_idx, Lz in enumerate(Lz_values):
    add_timing_row(
        "SARCOS",
        f"Proposed Lz={Lz}",
        sarcos_test_samples,
        sarcos_local + sarcos_protocol_times_by_lz[:, lz_idx],
    )
    add_timing_row(
        "Diabetes",
        f"Proposed Lz={Lz}",
        diab_test_samples,
        diab_local + diab_protocol_times_by_lz[:, lz_idx],
    )

total_online_time_table = pd.DataFrame(timing_rows)
display(
    total_online_time_table.style.format(
        {
            "mean_total_seconds": "{:.6f}",
            "std_total_seconds": "{:.6f}",
            "throughput_points_per_second": "{:.2f}",
        }
    )
)

for row in timing_rows:
    print(
        f"{row['dataset']:8s} | {row['configuration']:18s} | "
        f"all {row['test_points']} test points: "
        f"{row['mean_total_seconds']:.6f} +/- "
        f"{row['std_total_seconds']:.6f} s"
    )